# 02. Calidad de datos para los análisis

Consulta la evidencia de calidad generada por `src/preparar_analisis_bookeau.py` (paso `preparar`): demanda y agenda por segmento, casos estado–llegada para revisión, elegibilidad de la calificación y estratos comparables Express/Normal. Resultados en `docs/artifacts/calidad/`.

Las definiciones de los análisis y las reglas de negocio (R01–R07) están en el informe (`docs/entregables/latex/informe_latex.pdf`) y en el README.

**Antes de ejecutar:** desde la raíz del proyecto, correr `python3 src/main.py --dummy` (datos ficticios versionados) o `python3 src/main.py` (datos reales, que no se versionan). El cuaderno solo lee los resultados del pipeline en `docs/artifacts/`, `data/plata/` y `data/oro/`. Con datos ficticios los conteos son pequeños y no coinciden con las cifras del informe, que usan los datos reales.


In [1]:
from pathlib import Path
import csv
from html import escape
from IPython.display import display, HTML

PROYECTO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "main.py").is_file()),
    None,
)
if PROYECTO is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta del proyecto o desde notebooks/.")
RESULTADOS = PROYECTO / "docs" / "artifacts" / "calidad"
if not RESULTADOS.is_dir():
    raise RuntimeError("Faltan los resultados del pipeline. Ejecuta primero desde la raíz: "
                       "python3 src/main.py --dummy  (o sin --dummy si tienes los datos reales)")

def cargar(nombre):
    with (RESULTADOS / nombre).open(newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))

def tabla(filas, columnas=None, limite=None):
    if not filas:
        print("Sin registros para esta selección.")
        return
    columnas = columnas or list(filas[0])
    mostrar = filas if limite is None else filas[:limite]
    cabecera = "".join(f"<th>{escape(c)}</th>" for c in columnas)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{escape(str(fila[c]))}</td>" for c in columnas) + "</tr>"
        for fila in mostrar
    )
    display(HTML(f"<table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"))
    print(f"Mostrando {len(mostrar)} de {len(filas)} registros.")


## 1. Eventos de reserva y agenda

Los conteos incluyen espera, cancelaciones y otros estados. No equivalen a tutorías atendidas. La hora corresponde al inicio programado tal como se exporta; su zona horaria no está confirmada.


In [2]:
demanda = cargar("demanda_por_periodo_modalidad_servicio_estado.csv")
PERIODO = max(r["periodo"] for r in demanda)  # período más reciente; se puede fijar, p. ej. "202610"
MODALIDAD = "Express"
tabla([r for r in demanda if r["periodo"] == PERIODO and r["tipo_horario_original"] == MODALIDAD])
agenda = cargar("agenda_por_dia_hora.csv")
tabla([r for r in agenda if r["periodo"] == PERIODO and r["tipo_horario_original"] == MODALIDAD], limite=30)


periodo,tipo_horario_original,servicio_original,estado_reserva_original,eventos_reserva
202120,Express,IP,Cancelada a tiempo,10
202120,Express,IP,Cancelada con excusa,2
202120,Express,IP,Cola cancelada por reservación,4
202120,Express,IP,En cola,2
202120,Express,IP,Finalizada,19
202120,Express,IP,No asistió,3
202120,Express,IP,Realizada,2
202120,Express,IP,Reservada,2


Mostrando 8 de 8 registros.


periodo,tipo_horario_original,servicio_original,dia_semana_iso,hora_inicio,eventos_reserva
202120,Express,IP,1,8,1
202120,Express,IP,1,9,2
202120,Express,IP,1,13,1
202120,Express,IP,1,14,2
202120,Express,IP,1,15,3
202120,Express,IP,1,16,2
202120,Express,IP,1,17,1
202120,Express,IP,2,10,1
202120,Express,IP,2,11,2
202120,Express,IP,2,12,1


Mostrando 30 de 33 registros.


## 2. Estados y casos para revisión

49 eventos presentan combinaciones de estado y llegada que ameritan revisión: 8 Finalizada sin llegada, 9 No asistió con llegada y 32 estados de cancelación con llegada. No se corrigen ni excluyen automáticamente.


In [3]:
casos = cargar("casos_revision_estado_llegada.csv")
from collections import Counter
motivos = Counter(r["motivo_revision"] for r in casos)
tabla([{"motivo": motivo, "casos": n} for motivo, n in motivos.items()])
# Para revisar registros concretos con el responsable de la fuente:
# tabla(casos)


motivo,casos
Estado de cancelación con marca de llegada,4
Finalizada sin marca de llegada,8
No asistió con marca de llegada,1


Mostrando 3 de 3 registros.


## 3. Denominador de calificaciones según R02

El conteo utilizable se limita a respuestas marcadas Válida con calificación 1–5. Los otros estados y los faltantes se conservan en los resultados agregados. Completitud del ítem es la proporción de encuestas marcadas Válida que tiene calificación utilizable; no es tasa de respuesta entre todas las tutorías.


In [4]:
tabla(cargar("resumen_elegibilidad_satisfaccion.csv"))
satisfaccion = cargar("satisfaccion_por_periodo_modalidad_servicio.csv")
tabla([r for r in satisfaccion if r["periodo"] == PERIODO])


grupo,respuestas,marcadas_valida,otro_estado,calificaciones_validas_1_a_5,faltantes_en_validas,fuera_dominio_en_validas
Encuesta Express,168,165,3,105,60,0
Encuesta Normal,437,432,5,264,168,0


Mostrando 2 de 2 registros.


grupo_encuesta,periodo,tipo_horario_reserva,servicio_reserva,respuestas_exportadas,respuestas_marcadas_valida,respuestas_otro_estado,calificaciones_1_a_5_en_validas,sin_calificacion_en_validas,fuera_dominio_en_validas,completitud_item_en_validas_pct,media_descriptiva_provisional,mediana_descriptiva_provisional,n_calificacion_1,n_calificacion_2,n_calificacion_3,n_calificacion_4,n_calificacion_5
Encuesta Express,202120,"Deprecated, Grupal",IP,1,1,0,1,0,0,100.0,5,5,0,0,0,0,1
Encuesta Express,202120,Express,IP,13,13,0,12,1,0,92.3077,4.8333,5.0,0,0,0,2,10
Encuesta Normal,202120,Entrevistas,Entrevista,3,3,0,3,0,0,100.0,4,5,0,1,0,0,2
Encuesta Normal,202120,Normal,IP,36,36,0,33,3,0,91.6667,4.4848,5,2,1,2,2,26
Encuesta Normal,202120,Normal pico,IP,5,4,1,3,1,0,75.0,5,5,0,0,0,0,3
Encuesta Normal,202120,Sala,Sala,2,2,0,1,1,0,50.0,5,5,0,0,0,0,1


Mostrando 6 de 6 registros.


## 4. Estratos para comparar Express y Normal

Hay 37 combinaciones de período y servicio original con calificaciones en ambas modalidades exactas. Normal pico y modalidades Deprecated no se fusionan con Normal o Express. La igualdad de período, servicio y pregunta no garantiza equivalencia del formulario ni ausencia de sesgo de respuesta.


In [5]:
comparacion = cargar("estratos_comunes_express_normal.csv")
tabla(comparacion)


periodo,servicio_original,n_express,n_normal,media_express_provisional,media_normal_provisional,completitud_item_express_pct,completitud_item_normal_pct,limite
201819,"Deprecated, APO 1",2,5,5,4.6,66.6667,100.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201819,"Deprecated, APO 2",2,5,5,4.6,66.6667,83.3333,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201819,"Deprecated, DPOO",6,12,4.5,4.5,100.0,92.3077,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201819,EDA,4,8,4.25,4.5,100.0,88.8889,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201819,IP,4,10,5,4.6,100.0,90.9091,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, APO 1",2,4,5,4.75,66.6667,80.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,"Deprecated, APO 2",5,6,4.4,4.3333,100.0,75.0,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,EDA,4,10,4.75,4.7,100.0,83.3333,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201820,IP,7,14,4.4286,4.2143,100.0,87.5,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"
201920,IP,13,37,4.6923,4.3514,92.8571,92.5,"Misma pregunta/periodo/servicio; no garantiza equivalencia histórica, ausencia de sesgo ni efecto causal"


Mostrando 13 de 13 registros.


## 5. Regenerar las tablas

El script vuelve a leer todas las fuentes y comprueba unicidad y vínculos antes de agregar. Si cambian fuentes o reglas, revisar también las definiciones y la narrativa de los documentos.


In [6]:
# import runpy
# import sys
# sys.path.insert(0, str(PROYECTO / "src"))
# runpy.run_path(str(PROYECTO / "src" / "preparar_analisis_bookeau.py"), run_name="__main__")
